## INSTALL

In [2]:
import sys

REQUIRED_VERSION = (3, 11, 9)
current_version  = sys.version_info[:3]

if current_version != REQUIRED_VERSION:
    raise RuntimeError(
        f"Wrong Python version. Expected 3.11.9, "
        f"got {'.'.join(str(v) for v in current_version)}. "
        "Make sure the correct kernel is selected."
    )

print(f"Python {'.'.join(str(v) for v in current_version)} OK")

#%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
#%pip install pandas numpy Pillow tqdm scikit-learn xgboost timm
%pip install timm lightgbm catboost

Python 3.11.9 OK
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 9.5 MB/s  0:00:00
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
    --------------------------------------- 2.4/100.2 MB 12.2 MB/s eta 0:00:09
   - -------------------------------------- 4.7/100.2 MB 11.4 MB/s eta 0:00:09
   -- ------------------------------------- 6.8/100.2 MB 11.0 MB/s eta 0:00:09
   --- ------------------------------------ 9.2/100.2 MB 11.2 MB/s eta 0:00:09
   ---- ----------------------------------- 11.5/100.2 MB 11.3 MB/s eta 0:00:08
   ----- ---------------------------------- 13.9/100.2 MB 11.5 MB/s eta 0:00:08
   ------ --------------------------------- 16.5/100.2 MB 11.4 MB/s eta 0:00:08
   ------- -------------------------------- 18.9/100.2 MB 11.5 MB/s eta 0:00:08
   -------- ------------------------------- 21.5/100.2 MB


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## IMPORTS

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    StratifiedKFold, train_test_split,
    cross_val_score, ParameterGrid
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from itertools import product

## MODEL SELECTOR

In [4]:
import os

# Load HF_TOKEN from the .env file in the project root.
# This avoids hardcoding secrets in the notebook.
with open('.env') as env_file:
    for env_line in env_file:
        if '=' in env_line and not env_line.startswith('#'):
            env_key, env_value = env_line.strip().split('=', 1)
            os.environ[env_key] = env_value

print('HF_TOKEN loaded' if os.environ.get('HF_TOKEN') else 'WARNING: HF_TOKEN not found in .env')

HF_TOKEN loaded


In [5]:
# --- Model Selector -----------------------------------------------------------
# DINOv2 ViT-g/14 (Meta, 1536-d) — weights pre-trained on LVD-142M.
# Changing the backbone requires re-running PREPROCESSING and FEATURE EXTRACTION
# to regenerate the .pt feature files.
#
FEATURE_EXTRACTOR = 'dinov2'

# Cross-validation folds (used by HYPER PARAM SEARCH)
N_FOLDS = 5
# -----------------------------------------------------------------------------

## DEFINITIONS

In [12]:
# --- Device selection ---------------------------------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# --- Load DINOv2 ViT-g/14 backbone -------------------------------------------
# 1536-d feature vectors; weights pre-trained on LVD-142M.
backbone_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
backbone_model = backbone_model.to(device).eval()

backbone_preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def encode(batch):
    return backbone_model(batch.to(device)).float().cpu()

feat_suffix = '_dinov2'
print(f'Backbone: dinov2  |  suffix: {feat_suffix!r}  |  device: {device}')


# --- Image loading and feature extraction helpers ----------------------------

def save_images(image_dir, output_file, has_labels=True):
    """Load images from image_dir, apply backbone_preprocess, save tensors to output_file.
    has_labels=True  : folder has one subfolder per class (training set).
    has_labels=False : flat folder of images (test set)."""
    image_tensors = []
    labels        = []
    file_paths    = []

    if has_labels:
        class_names    = sorted(os.listdir(image_dir))
        class_to_index = {name: i for i, name in enumerate(class_names)}
        total_images   = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )
        count = 0
        for class_name in class_names:
            for filename in os.listdir(os.path.join(image_dir, class_name)):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                image_path = os.path.join(image_dir, class_name, filename)
                image_tensors.append(backbone_preprocess(Image.open(image_path).convert('RGB')))
                labels.append(class_to_index[class_name])
                file_paths.append(image_path)
                count += 1
                print(f'{count}/{total_images}', end='\r')
        torch.save({
            'tensors':        torch.stack(image_tensors),
            'labels':         torch.tensor(labels),
            'paths':          file_paths,
            'class_to_index': class_to_index,
        }, output_file)
    else:
        image_files  = [f for f in os.listdir(image_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        total_images = len(image_files)
        for i, filename in enumerate(image_files, 1):
            image_path = os.path.join(image_dir, filename)
            image_tensors.append(backbone_preprocess(Image.open(image_path).convert('RGB')))
            file_paths.append(image_path)
            print(f'{i}/{total_images}', end='\r')
        torch.save({'tensors': torch.stack(image_tensors), 'paths': file_paths}, output_file)

    print(f'\nSaved {len(image_tensors)} images to {output_file}')


def extract(input_file, output_file):
    """Pass every image tensor through the DINOv2 backbone; save feature vectors."""
    data         = torch.load(input_file, weights_only=False)
    images       = data['tensors']
    total_images = len(images)
    feature_list = []
    with torch.no_grad():
        for i, image in enumerate(images, 1):
            feature_list.append(encode(image.unsqueeze(0)).squeeze().numpy())
            print(f'{i}/{total_images}', end='\r')
    features = np.array(feature_list)
    torch.save({**data, 'features': features}, output_file)
    print(f'\nExtracted {features.shape} -> saved to {output_file}')

Device: cuda


Using cache found in C:\Users\TheRe/.cache\torch\hub\facebookresearch_dinov2_main


Backbone: dinov2  |  suffix: '_dinov2'  |  device: cuda


In [13]:
# Metrics

def compute_classification_metrics(true_labels, predicted_labels):
    # Class distribution
    unique_classes, samples_per_class = np.unique(true_labels, return_counts=True)

    class_distribution_percentages = {
        int(class_index): round(100 * class_sample_count / len(true_labels), 1)
        for class_index, class_sample_count in zip(unique_classes, samples_per_class)
    }

    # all other metrics
    return {
        'accuracy':      accuracy_score(true_labels, predicted_labels),
        'f1':            f1_score(true_labels, predicted_labels, average='weighted'),
        'precision':     precision_score(true_labels, predicted_labels, average='weighted'),
        'recall':        recall_score(true_labels, predicted_labels, average='weighted'),
        'class_balance': class_distribution_percentages,
    }


def print_cross_validation_report(model_name, fold_metrics_list):
    # Report mean +- std of performance across the folds
    print(f'--- {model_name} ---')

    for metric_name in ('accuracy', 'f1', 'precision', 'recall'):
        metric_values_per_fold = [fold_result[metric_name] for fold_result in fold_metrics_list]
        
        mean_value = np.mean(metric_values_per_fold)
        std_value  = np.std(metric_values_per_fold)
        
        print(f'  {metric_name:<9}: {mean_value:.4f} +/- {std_value:.4f}')

    print('  class balance:')
    for class_index, percentage in fold_metrics_list[0]['class_balance'].items():
        print(f'    class {class_index}: {percentage}%')

    print()


# --- MLP model ---------------------------------------------------------------

class MLP(nn.Module):
    # we can experiment with different layer setups and activations and dropouts
    def __init__(self, input_size, num_classes, hidden_layers=(256, 128),
                 activation=nn.ReLU, dropout=0.0):
        super().__init__()

        network_layers     = []
        current_layer_size = input_size

        for layer_output_size in hidden_layers:
            
            network_layers.append(nn.Linear(current_layer_size, layer_output_size))
            network_layers.append(activation())

            if dropout > 0:
                network_layers.append(nn.Dropout(dropout))

            current_layer_size = layer_output_size

        network_layers.append(nn.Linear(current_layer_size, num_classes))
        self.network = nn.Sequential(*network_layers)

    def forward(self, input_tensor):
        return self.network(input_tensor)

# --- Classifier training functions -------------------------------------------
# All functions accept NumPy arrays and return (trained_model, metrics_dict).

def train_logistic_regression(training_features, training_labels,
                               validation_features, validation_labels,
                               regularisation_strength=1.0):
    
    # create and fit model
    model = LogisticRegression(C=regularisation_strength, max_iter=2000, random_state=42)
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    
    # predict the val set
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

def train_support_vector_machine(training_features, training_labels,
                                  validation_features, validation_labels,
                                  kernel='rbf', regularisation_strength=1.0,
                                  gamma='scale'):
    
    # create and fit model
    model = SVC(
        kernel=kernel,
        C=regularisation_strength,
        gamma=gamma,
        decision_function_shape='ovr',
        random_state=42,
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    
    # predict the val set
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

def train_random_forest(training_features, training_labels,
                         validation_features, validation_labels,
                         number_of_trees=300):
    
    # create and fit model
    model = RandomForestClassifier(
        n_estimators=number_of_trees,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    )
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    
    # predict the val set
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

def train_xgboost_classifier(training_features, training_labels,
                              validation_features, validation_labels):

    # create and fit model
    model = XGBClassifier(
        objective='multi:softmax', # use softmax for multi-class classification
        eval_metric='mlogloss', # use log-loss as the evaluation metric
        verbosity=0,
        random_state=42,
    )
    
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    
    # predict the val set
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

def train_multilayer_perceptron(training_features, training_labels,
                                 validation_features, validation_labels,
                                 hidden_layers=(256, 128), activation=nn.ReLU,
                                 dropout=0.0, epochs=200, learning_rate=1e-3):
    # GPU preferred but CPU works
    
    compute_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
    number_of_classes = len(np.unique(training_labels))

    # convert data to PyTorch tensors and move to device
    training_input     = torch.tensor(np.asarray(training_features,   np.float32)).to(compute_device)
    training_targets   = torch.tensor(np.asarray(training_labels,     np.int64)).to(compute_device)
    
    validation_input   = torch.tensor(np.asarray(validation_features, np.float32)).to(compute_device)
    validation_targets = torch.tensor(np.asarray(validation_labels,   np.int64)).to(compute_device)

    # we establish the batchsizing and shuffling here
    training_data_loader = DataLoader(
        TensorDataset(training_input, training_targets),
        batch_size=256,
        shuffle=True
    )

    torch.manual_seed(42)
    
    # create model, optimizer, and loss function
    model         = MLP(training_input.shape[1], number_of_classes, hidden_layers, activation, dropout).to(compute_device)
    optimizer     = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    loss_function = nn.CrossEntropyLoss()

    # Early stopping
    best_validation_loss       = float('inf')
    best_model_weights         = None
    epochs_without_improvement = 0

    # train loop
    for epoch in range(epochs):
        model.train() 

        # iterate over batches
        for input_batch, target_batch in training_data_loader:
            
            optimizer.zero_grad() # clear gradients from previous step
            batch_loss = loss_function(model(input_batch), target_batch) # compute loss for the batch
            batch_loss.backward() # backpropagate to compute gradients
            optimizer.step() # update model weights based on gradients

        
        model.eval()

        # time to evaluate current performance
        with torch.no_grad():
            # get loss
            current_validation_loss = loss_function(model(validation_input), validation_targets).item()

        # did we improve over the best epoch so far?
        if current_validation_loss < best_validation_loss:
            best_validation_loss       = current_validation_loss
            best_model_weights         = model.state_dict()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= 25:
                break

    model.load_state_dict(best_model_weights)
    model.eval()

    # predict the val set using best weights
    with torch.no_grad():
        predictions = model(validation_input).argmax(dim=1).cpu().numpy()

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )
      
def train_knn_classifier(training_features, training_labels,
                          validation_features, validation_labels,
                          number_of_neighbours=10, distance_metric='cosine'):
    
    # according to the papers, both of the models have embeddings lie on a spherical like space
    # so we use cosine as defaullt
    
    # create model and fit
    model = KNeighborsClassifier(
        n_neighbors=number_of_neighbours,
        metric=distance_metric,
        n_jobs=-1
    )
    
    model.fit(
        np.asarray(training_features, np.float64),
        np.asarray(training_labels, np.int64)
    )
    
    predictions = model.predict(np.asarray(validation_features, np.float64))

    return model, compute_classification_metrics(
        np.asarray(validation_labels, np.int64), predictions
    )

In [ ]:
# --- Fine-tune helpers -------------------------------------------------------

def _freeze_and_unfreeze_backbone(num_blocks):
    """Freeze all backbone params, then unfreeze the last num_blocks transformer
    blocks and the final norm layer.
    Returns (blocks, norm, actual_num_blocks)."""
    blocks = list(backbone_model.blocks)   # 40 transformer blocks for ViT-g/14
    norm   = backbone_model.norm
    if num_blocks is None:
        num_blocks = 4   # DINOv2 paper recommendation: 4 of 40 blocks
    for param in backbone_model.parameters():
        param.requires_grad = False
    for block in blocks[-num_blocks:]:
        for param in block.parameters():
            param.requires_grad = True
    for param in norm.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in backbone_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in backbone_model.parameters())
    print(f'Trainable: {trainable:,}/{total:,} ({100*trainable/total:.1f}%)'
          f'  [{num_blocks}/{len(blocks)} blocks unfrozen]')
    return blocks, norm, num_blocks


def _build_prefix_and_suffix(blocks, num_blocks):
    """Return (frozen_prefix_fn, suffix_fn) for DINOv2.

    frozen_prefix_fn(batch) : patch-embed + all frozen transformer blocks.
    suffix_fn(x)            : unfrozen blocks + norm -> CLS token (B, 1536).
    """
    def frozen_prefix_fn(batch):
        x = backbone_model.prepare_tokens_with_masks(batch)
        for block in backbone_model.blocks[:-num_blocks]:
            x = block(x)
        return x

    def suffix_fn(x):
        for block in blocks[-num_blocks:]:
            x = block(x)
        x = backbone_model.norm(x)
        return x[:, 0]   # CLS token -> (B, 1536)

    return frozen_prefix_fn, suffix_fn


def _cache_frozen_activations(images, frozen_prefix_fn, batch_size):
    """Pre-compute frozen-prefix activations for all images in fp16.
    Stores cache on CPU to save VRAM.  Returns tensor of shape (N, seq_len, d)."""
    with torch.no_grad():
        sample = frozen_prefix_fn(images[:1].to(device))
    est_gb = sample.numel() * len(images) * 2 / 1e9
    print(f'Caching frozen activations (~{est_gb:.1f} GB in fp16)...')
    chunks = []
    backbone_model.eval()
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            chunks.append(
                frozen_prefix_fn(images[i:i+batch_size].to(device)).cpu().half()
            )
            print(f'  {min(i+batch_size, len(images))}/{len(images)}', end='\r')
    cache = torch.cat(chunks)
    print(f'\n  Done. Cache shape: {tuple(cache.shape)}  (fp16)')
    return cache


def _extract_via_suffix(source, suffix_fn, use_cache, batch_size):
    """Run source through suffix_fn (or full backbone when not caching) in batches.
    Returns a CPU float32 feature tensor."""
    chunks = []
    with torch.no_grad():
        for i in range(0, len(source), batch_size):
            chunk = source[i:i+batch_size].to(device=device, dtype=torch.float32)
            chunks.append(
                (suffix_fn(chunk) if use_cache else backbone_model(chunk)).cpu()
            )
    return torch.cat(chunks)


def _build_finetune_head(source_sample, suffix_fn, use_cache, num_classes, dropout):
    """Build a 2-layer (Linear -> GELU -> Dropout -> Linear) classification head.
    Infers feature dimension from a single forward pass through source_sample."""
    with torch.no_grad():
        sample   = source_sample[:1].to(device).float()
        feat_dim = (suffix_fn(sample) if use_cache else backbone_model(sample)).shape[-1]
    return nn.Sequential(
        nn.Linear(feat_dim, 512), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    ).to(device)


def _run_finetune_training_loop(head, train_loader, val_source, labels, val_idx,
                                suffix_fn, use_cache, optimizer, loss_fn,
                                grad_scaler, batch_dtype, use_amp,
                                epochs, patience, trainable_param_names, batch_size):
    """Train backbone + head with early stopping.
    Returns (best_backbone_state_dict, best_head_state_dict)."""
    best_val_loss    = float('inf')
    best_bb_state    = None
    best_head_state  = None
    no_improve_count = 0

    for epoch in range(epochs):
        backbone_model.train()
        head.train()
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device=device, dtype=batch_dtype)
            y_batch = y_batch.to(device)
            optimizer.zero_grad()
            with torch.autocast('cuda', enabled=use_amp):
                feats = suffix_fn(x_batch) if use_cache else backbone_model(x_batch)
                loss  = loss_fn(head(feats), y_batch)
            grad_scaler.scale(loss).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()

        backbone_model.eval()
        head.eval()
        val_feats = _extract_via_suffix(val_source, suffix_fn, use_cache, batch_size).to(device)
        val_loss  = loss_fn(head(val_feats), labels[val_idx].to(device)).item()
        print(f'Epoch {epoch+1:>3}/{epochs}  val_loss={val_loss:.4f}', end='\r')

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            best_bb_state  = {
                k: v.cpu().clone()
                for k, v in backbone_model.state_dict().items()
                if k in trainable_param_names
            }
            best_head_state  = {k: v.cpu().clone() for k, v in head.state_dict().items()}
            no_improve_count = 0
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break

    print(f'\nBest val_loss: {best_val_loss:.4f}')
    return best_bb_state, best_head_state


# --- Main fine-tune entry point ----------------------------------------------

def finetune(images_file, train_idx, val_idx, num_classes,
             num_blocks=None, backbone_lr=1e-5, head_lr=1e-3,
             dropout=0.1, epochs=20, batch_size=32,
             patience=5, cache_frozen=True):
    """Fine-tune the last num_blocks DINOv2 transformer blocks on the training split.

    cache_frozen=True  Pre-compute frozen activations once (~3 GB fp16) for a
                       ~5-10x training speed-up.  Set False only if still OOM.
    num_blocks=None    Uses the DINOv2 paper default: 4 of 40 blocks.
    """
    print(f'Device: {device}  |  backbone: dinov2  (GPU strongly recommended)')

    # mmap=True: pages read from disk on demand — much lower peak RAM.
    data   = torch.load(images_file, weights_only=False, mmap=True)
    images = data['tensors']
    labels = data['labels']

    # Step 1: freeze backbone, unfreeze last num_blocks + norm
    blocks, norm, num_blocks = _freeze_and_unfreeze_backbone(num_blocks)

    # Step 2: build frozen-prefix and trainable-suffix functions
    frozen_prefix_fn, suffix_fn = _build_prefix_and_suffix(blocks, num_blocks)

    # Step 3: cache frozen activations (fast path) or use raw images (slow path)
    if cache_frozen:
        frozen_cache  = _cache_frozen_activations(images, frozen_prefix_fn, batch_size)
        del data, images   # free RAM — frozen_cache holds everything we need
        train_dataset = TensorDataset(frozen_cache[train_idx], labels[train_idx])
        val_source    = frozen_cache[val_idx]
    else:
        train_dataset = TensorDataset(images[train_idx], labels[train_idx])
        val_source    = images[val_idx]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Step 4: build classification head (feature dim auto-detected)
    head = _build_finetune_head(
        frozen_cache if cache_frozen else images,
        suffix_fn, cache_frozen, num_classes, dropout,
    )

    # Step 5: optimizer, loss function, AMP scaler
    use_amp     = (device == 'cuda')
    optimizer   = torch.optim.AdamW([
        {'params': [p for p in backbone_model.parameters() if p.requires_grad], 'lr': backbone_lr},
        {'params': head.parameters(), 'lr': head_lr},
    ], weight_decay=1e-4)
    loss_fn     = nn.CrossEntropyLoss()
    grad_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    batch_dtype = torch.float16 if use_amp else torch.float32
    trainable_param_names = {
        name for name, p in backbone_model.named_parameters() if p.requires_grad
    }

    # Step 6: training loop with early stopping
    best_bb_state, best_head_state = _run_finetune_training_loop(
        head, train_loader, val_source, labels, val_idx,
        suffix_fn, cache_frozen, optimizer, loss_fn,
        grad_scaler, batch_dtype, use_amp,
        epochs, patience, trainable_param_names, batch_size,
    )

    # Step 7: restore best weights, compute final val predictions, re-freeze
    bb_state = backbone_model.state_dict()
    bb_state.update({k: v.to(device) for k, v in best_bb_state.items()})
    backbone_model.load_state_dict(bb_state)
    head.load_state_dict({k: v.to(device) for k, v in best_head_state.items()})

    backbone_model.eval()
    head.eval()
    val_feats = _extract_via_suffix(val_source, suffix_fn, cache_frozen, batch_size).to(device)
    val_preds = head(val_feats).argmax(dim=1).cpu().numpy()

    for param in backbone_model.parameters():
        param.requires_grad = False

    return head, compute_classification_metrics(labels[val_idx].numpy(), val_preds)

: 

## PREPROCESSING
Load images from disk, apply the whatever model's transform, and save as tensors.

In [ ]:
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'

if not os.path.exists(train_image_dir):
    raise FileNotFoundError(f'Training folder not found: {train_image_dir}')
if not os.path.exists(test_image_dir):
    raise FileNotFoundError(f'Test folder not found: {test_image_dir}')

# Save preprocessed tensors
save_images(train_image_dir, f'Data\\task1_data\\t1_train_transformed{feat_suffix}.pt', has_labels=True)
save_images(test_image_dir,  f'Data\\task1_data\\t1_test_transformed{feat_suffix}.pt',  has_labels=False)

3750/3750
Saved 3750 images to Data\task1_data\t1_train_transformed_convnextv2.pt
1250/1250
Saved 1250 images to Data\task1_data\t1_test_transformed_convnextv2.pt


## FEATURE EXTRACTION
Load raw images and run them through the DINOv2 backbone.
Only needs to run once; re-run only if the image files change.

In [10]:
# Helper functions for loading images and extracting DINOv2 features.
# Called by the extraction loop in the next cell.

compute_device = 'cuda' if torch.cuda.is_available() else 'cpu'

# DINOv2 preprocessing: standard ImageNet stats, 224 px centre crop.
dinov2_image_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def load_images_from_dir(image_dir, image_transform, has_labels):
    """Load and preprocess all images from image_dir.
    has_labels=True  : folder contains one subfolder per class.
    has_labels=False : flat folder (test set).
    Returns (image_tensors, label_tensor_or_None, file_paths, class_to_index)."""
    image_tensors_list = []
    label_list         = []
    file_paths         = []
    class_to_index     = {}

    if has_labels:
        class_names    = sorted(os.listdir(image_dir))
        class_to_index = {cls: i for i, cls in enumerate(class_names)}
        total_images   = sum(
            len(os.listdir(os.path.join(image_dir, cls)))
            for cls in class_names
            if os.path.isdir(os.path.join(image_dir, cls))
        )
        image_count = 0
        for class_name in class_names:
            for filename in os.listdir(os.path.join(image_dir, class_name)):
                if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                    continue
                image_path = os.path.join(image_dir, class_name, filename)
                image_tensors_list.append(
                    image_transform(Image.open(image_path).convert('RGB'))
                )
                label_list.append(class_to_index[class_name])
                file_paths.append(image_path)
                image_count += 1
                print(f'  {image_count}/{total_images}', end='\r')
    else:
        image_files = sorted(
            fn for fn in os.listdir(image_dir)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png'))
        )
        for i, filename in enumerate(image_files, 1):
            image_path = os.path.join(image_dir, filename)
            image_tensors_list.append(
                image_transform(Image.open(image_path).convert('RGB'))
            )
            file_paths.append(image_path)
            print(f'  {i}/{len(image_files)}', end='\r')

    label_tensor = torch.tensor(label_list) if label_list else None
    return torch.stack(image_tensors_list), label_tensor, file_paths, class_to_index


def extract_and_save_features(backbone_instance, encode_fn, image_tensors,
                               label_tensor, file_paths, class_to_index,
                               output_file, batch_size=32):
    """Run image_tensors through backbone_instance in batches; save to output_file.
    Uses AMP (fp16) on CUDA for faster throughput."""
    total_images   = len(image_tensors)
    feature_chunks = []
    use_amp        = (compute_device == 'cuda')

    with torch.no_grad():
        for start in range(0, total_images, batch_size):
            batch = image_tensors[start : start + batch_size].to(compute_device)
            with torch.autocast(device_type=compute_device, enabled=use_amp):
                feats = encode_fn(backbone_instance, batch)
            feature_chunks.append(feats.float().cpu())
            print(f'  {min(start + batch_size, total_images)}/{total_images}', end='\r')

    all_features = torch.cat(feature_chunks).numpy()
    save_dict    = {'tensors': image_tensors, 'paths': file_paths, 'features': all_features}
    if label_tensor is not None:
        save_dict['labels']         = label_tensor
        save_dict['class_to_index'] = class_to_index
    torch.save(save_dict, output_file)
    print(f'\n  Saved {all_features.shape} -> {output_file}')

c:\Users\TheRe\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Main DINOv2 feature extraction ----------------------------------------------
train_image_dir = r'Data\task1_data\images\train'
test_image_dir  = r'Data\task1_data\images\test'


def _run_dinov2_extraction():
    """Load DINOv2, extract train and test features, save as .pt files."""
    print(f'=== DINOv2 ViT-g/14 (device={compute_device}) ===')
    model     = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
    model     = model.to(compute_device).eval()
    encode_fn = lambda m, batch: m(batch).float()

    print('  train images:')
    t_imgs, t_lbls, t_paths, t_c2i = load_images_from_dir(
        train_image_dir, dinov2_image_transform, has_labels=True
    )
    print()
    extract_and_save_features(
        model, encode_fn, t_imgs, t_lbls, t_paths, t_c2i,
        r'Data\task1_data\t1_train_features_dinov2.pt',
    )

    print('  test images:')
    e_imgs, _, e_paths, _ = load_images_from_dir(
        test_image_dir, dinov2_image_transform, has_labels=False
    )
    print()
    extract_and_save_features(
        model, encode_fn, e_imgs, None, e_paths, {},
        r'Data\task1_data\t1_test_features_dinov2.pt',
    )

    del model
    if compute_device == 'cuda':
        torch.cuda.empty_cache()
    print('\nDone. Feature files ready.')


_run_dinov2_extraction()

## FINE-TUNING
Optionally fine-tune the last few DINOv2 transformer blocks on your data,
then re-extract features with the adapted backbone.
Skip this section if you want to use frozen features only.

In [ ]:

raw_pt      = torch.load(f'Data\\task1_data\\t1_train_features{feat_suffix}.pt', weights_only=False) # the vector embeddings of the images
num_classes = len(np.unique(raw_pt['labels'].numpy()))
all_idx     = np.arange(len(raw_pt['labels']))

# split train/val
train_idx_ft, val_idx_ft = train_test_split(
    all_idx, test_size=0.2, random_state=42,
    stratify=raw_pt['labels'].numpy()
)

fine_tuned_head, ft_metrics = finetune(
    images_file = f'Data\\task1_data\\t1_train_features{feat_suffix}.pt',
    train_idx   = train_idx_ft,
    val_idx     = val_idx_ft,
    num_classes = num_classes,
    num_blocks  = None,
    backbone_lr = 1e-5,
    head_lr     = 1e-3,
    dropout     = 0.1,
    epochs      = 20,
    batch_size  = 8,    # be careful with this, depending on RAM/VRAM you could either crash the cell or spend an eternity waiting             
    cache_frozen=True, # just keep this on, its a godsend
)

print(f'Fine-tuned val:  F1={ft_metrics["f1"]:.4f}  Acc={ft_metrics["accuracy"]:.4f}')
print('\nRe-extracting features with fine-tuned backbone...')

extract(f'Data\\task1_data\\t1_train_features{feat_suffix}.pt', f'Data\\task1_data\\t1_train_features_ft{feat_suffix}.pt')
extract(f'Data\\task1_data\\t1_test_features{feat_suffix}.pt',  f'Data\\task1_data\\t1_test_features_ft{feat_suffix}.pt')

print('Done. Set USE_FINETUNED = True in the LOAD DATA cell to use these features.')

Device: cuda  |  backbone: dinov2  (GPU strongly recommended)
Trainable: 113,349,632/1,136,480,768 (10.0%)  [4/40 blocks unfrozen]


KeyboardInterrupt: 

## LOAD DATA

In [15]:
# Set USE_FINETUNED = True after running the FINE-TUNING cell to use the adapted features.
USE_FINETUNED = False

if USE_FINETUNED:
    train_data = torch.load(f'Data\\task1_data\\t1_train_features_ft{feat_suffix}.pt', weights_only=False)
    test_data  = torch.load(f'Data\\task1_data\\t1_test_features_ft{feat_suffix}.pt',  weights_only=False)
else:
    train_data = torch.load(f'Data\\task1_data\\t1_train_features{feat_suffix}.pt', weights_only=False)
    test_data  = torch.load(f'Data\\task1_data\\t1_test_features{feat_suffix}.pt',  weights_only=False)

train_features = train_data['features']        # shape: (N, D)
train_labels   = train_data['labels'].numpy()  # shape: (N,)
test_features  = test_data['features']         # shape: (M, D) — no labels

print(f'Train: {train_features.shape}  Labels: {train_labels.shape}')
print(f'Test (submission): {test_features.shape}')
print(f'Classes: {train_data["class_to_index"]}')

# Stratified 80/20 split, stratify ensures class proportions are preserved in both halves.
# while the data is already even in balance, its good practice to account for
(   training_split_features,
    validation_split_features,
    training_split_labels,
    validation_split_labels,
) = train_test_split(
    train_features,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

print(f'\nTraining split:   {training_split_features.shape}')
print(f'Validation split: {validation_split_features.shape}')

Train: (3750, 1536)  Labels: (3750,)
Test (submission): (1250, 1536)
Classes: {'bird': 0, 'butterfly': 1, 'cat': 2, 'deer': 3, 'dog': 4, 'elephant': 5, 'frog': 6, 'horse': 7, 'sheep': 8, 'spider': 9}

Training split:   (3000, 1536)
Validation split: (750, 1536)


## HYPER PARAM SEARCH

In [ ]:
# =============================================================================
# HYPER PARAM SEARCH  (N_FOLDS-fold OOF cross-validation)
# =============================================================================
# Each candidate is evaluated with N_FOLDS-fold stratified CV on the full
# training set.  sklearn estimators are cloned per fold so folds never share
# state.  MLP candidates are passed as dicts and call train_multilayer_perceptron
# per fold.
#
# Prerequisite: run LOAD DATA first (populates train_features / train_labels).

SEARCH_LOGREG = True
SEARCH_KNN    = True
SEARCH_SVM    = True
SEARCH_MLP    = False   # expensive — disable for quick runs

from sklearn.base import clone


# --- Core OOF evaluator ------------------------------------------------------

def cross_val_f1(model_or_params, features, labels, n_folds=N_FOLDS):
    """N-fold stratified OOF cross-validation.

    sklearn estimators are cloned per fold (no shared state).
    MLP candidates: pass a dict with keys 'type'='mlp', 'hidden', 'activation',
                    'dropout', 'lr'.
    Returns (mean_f1, std_f1) across all folds.
    """
    kfold    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_f1s = []

    for fold_train_idx, fold_val_idx in kfold.split(features, labels):
        fold_train     = features[fold_train_idx]
        fold_val       = features[fold_val_idx]
        fold_train_lbl = labels[fold_train_idx]
        fold_val_lbl   = labels[fold_val_idx]

        if isinstance(model_or_params, dict) and model_or_params.get('type') == 'mlp':
            p = model_or_params
            _, met = train_multilayer_perceptron(
                fold_train, fold_train_lbl, fold_val, fold_val_lbl,
                hidden_layers = p['hidden'],
                activation    = p.get('activation', nn.ReLU),
                dropout       = p.get('dropout',    0.0),
                learning_rate = p.get('lr',         1e-3),
            )
            fold_f1s.append(met['f1'])
        else:
            m = clone(model_or_params)
            m.fit(np.asarray(fold_train), np.asarray(fold_train_lbl))
            preds = m.predict(np.asarray(fold_val))
            fold_f1s.append(
                f1_score(fold_val_lbl, preds, average='weighted', zero_division=0)
            )

    return float(np.mean(fold_f1s)), float(np.std(fold_f1s))


# --- Build candidate list ----------------------------------------------------

def build_search_candidates():
    """Build (label, model_or_params) pairs for all enabled model families."""
    candidates = []

    if SEARCH_LOGREG:
        for C in [0.001, 0.01, 0.1, 1, 10, 100]:
            candidates.append((
                f'LogReg C={C}',
                LogisticRegression(C=C, max_iter=2000, random_state=42),
            ))

    if SEARCH_KNN:
        for k in [5, 10, 15, 20]:
            for metric in ['cosine', 'euclidean']:
                candidates.append((
                    f'kNN k={k} {metric}',
                    KNeighborsClassifier(n_neighbors=k, metric=metric, n_jobs=1),
                ))

    if SEARCH_SVM:
        svm_grid = [
            {'kernel': ['linear'],  'C': [0.001, 0.01, 0.1, 1, 10, 100]},
            {'kernel': ['rbf'],     'C': [0.01, 0.1, 1, 10, 100],
                                      'gamma': ['scale', 'auto', 0.01]},
            {'kernel': ['poly'],    'C': [0.01, 0.1, 1, 10],
                                      'gamma': ['scale', 'auto'],
                                      'degree': [2, 3]},
            {'kernel': ['sigmoid'], 'C': [0.01, 0.1, 1, 10],
                                      'gamma': ['scale', 'auto', 0.01]},
        ]
        for p in ParameterGrid(svm_grid):
            label = '  '.join(f'{k}={v}' for k, v in sorted(p.items()))
            candidates.append((
                f'SVM {label}',
                SVC(decision_function_shape='ovr', random_state=42, **p),
            ))

    if SEARCH_MLP:
        for hidden in [(1024, 256), (512, 256), (256, 128)]:
            for act in [nn.ReLU, nn.GELU]:
                for drop in [0.0, 0.4]:
                    for lr in [1e-3, 1e-4]:
                        candidates.append((
                            f'MLP {hidden} act={act.__name__} drop={drop} lr={lr}',
                            {'type': 'mlp', 'hidden': hidden, 'activation': act,
                             'dropout': drop, 'lr': lr},
                        ))

    return candidates


# --- Run search and collect results ------------------------------------------

def run_hyperparameter_search(features, labels, candidates):
    """Evaluate every candidate with N_FOLDS-fold OOF CV.
    Prints running progress and returns a DataFrame sorted by mean_f1 descending."""
    results = []
    n_cands = len(candidates)
    print(f'\nSearching {n_cands} candidates ({len(labels)} samples, {N_FOLDS}-fold CV)')

    for i, (label, model_or_params) in enumerate(candidates, 1):
        mean_f1, std_f1 = cross_val_f1(model_or_params, features, labels)
        print(f'  [{i:>3}/{n_cands}] {label:<55}  F1={mean_f1:.4f} +/- {std_f1:.4f}')
        results.append({'model': label, 'mean_f1': mean_f1, 'std_f1': std_f1})

    return pd.DataFrame(results).sort_values('mean_f1', ascending=False)


# =============================================================================
# Run the search
# =============================================================================

all_features = np.asarray(train_features)
all_labels   = np.asarray(train_labels)

candidates  = build_search_candidates()
results_df  = run_hyperparameter_search(all_features, all_labels, candidates)

fine_tuning_suffix = 'ft' if USE_FINETUNED else 'noft'
csv_path = f'Data/task1_data/search_results_{FEATURE_EXTRACTOR}_{fine_tuning_suffix}.csv'
results_df.to_csv(csv_path, index=False)

print(f'\nTop 20 results:')
print(results_df.head(20).to_string(index=False))
print(f'\nFull results saved to {csv_path}')

Logistic Regression search: 6 configs


LogReg: 100%|██████████| 6/6 [00:00<?, ?it/s]



kNN search: 8 configs


kNN: 100%|██████████| 8/8 [00:00<00:00, 7998.67it/s]



SVM search: 101 configs


SVM: 100%|██████████| 101/101 [02:23<00:00,  1.42s/it]



MLP search: 468 configs


MLP: 100%|██████████| 468/468 [06:48<00:00,  1.15it/s]

Saved full results to Data/task1_data/hyperparameter_search_results_dinov2_noft.csv

   TOP 100 — 583 configs evaluated, ranked by weighted F1
   #    Model       F1        Params
   --------------------------------------------------------------------------------------
   1    MLP         0.9920    layers=(1024, 256)  act=ReLU  drop=0.4  lr=0.0001
   2    LogReg      0.9907    C=0.01
   3    MLP         0.9907    layers=(1024, 256, 256)  act=GELU  drop=0.0  lr=0.0001
   4    MLP         0.9907    layers=(1024, 256, 256)  act=GELU  drop=0.2  lr=0.0001
   5    MLP         0.9907    layers=(512, 1024, 512)  act=ReLU  drop=0.4  lr=0.0001
   6    MLP         0.9907    layers=(256,)  act=ReLU  drop=0.4  lr=0.0001
   7    MLP         0.9907    layers=(1024, 256, 256)  act=ReLU  drop=0.2  lr=0.0001
   8    MLP         0.9907    layers=(256, 256, 512)  act=ReLU  drop=0.2  lr=0.0001
   9    MLP         0.9907    layers=(1024,)  act=GELU  drop=0.2  lr=0.0001
   10   LogReg      0.9907    C=0.1
  

## SUBMISSION

In [16]:
# --- Settings ----------------------------------------------------------------
# Choose a model and paste in the best params from the search above.
SUBMISSION_MODEL = 'mlp'   # 'linear' | 'svm' | 'knn' | 'xgboost' | 'lgbm' | 'catboost' | 'mlp' | 'stack'

SVM_SUBMISSION_PARAMS        = {'kernel': 'rbf', 'C': 1, 'gamma': 0.01}
KNN_NEIGHBOURS               = 10
KNN_DISTANCE_METRIC          = 'cosine'
LIGHTGBM_SUBMISSION_PARAMS   = {'n_estimators': 500, 'num_leaves': 63, 'learning_rate': 0.05}
CATBOOST_SUBMISSION_PARAMS   = {'iterations': 500, 'depth': 6, 'learning_rate': 0.05}

MLP_SUBMISSION_HIDDEN_LAYERS  = (1024, 256)
MLP_SUBMISSION_DROPOUT_RATE   = 0.4
MLP_SUBMISSION_EPOCHS         = 500
MLP_SUBMISSION_LEARNING_RATE  = 0.0001
MLP_ACTIVATION = nn.ReLU
# -----------------------------------------------------------------------------

all_training_features    = np.asarray(train_features, np.float64)
all_training_labels      = np.asarray(train_labels,   np.int64)
test_submission_features = np.asarray(test_features,  np.float64)

print(f'Training {SUBMISSION_MODEL} on all {len(all_training_features)} labelled samples...')

if SUBMISSION_MODEL == 'linear':
    model = LogisticRegression(max_iter=2000, random_state=42)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'svm':
    model = SVC(decision_function_shape='ovr', random_state=42, **SVM_SUBMISSION_PARAMS)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'knn':
    model = KNeighborsClassifier(n_neighbors=KNN_NEIGHBOURS, metric=KNN_DISTANCE_METRIC, n_jobs=-1)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'xgboost':
    model = XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', verbosity=0, random_state=42)
    model.fit(all_training_features, all_training_labels)
    predictions = model.predict(test_submission_features)

elif SUBMISSION_MODEL == 'mlp':
    # Hold out 5% for early stopping
    holdout_training_features, holdout_validation_features, holdout_training_labels, holdout_validation_labels = train_test_split(
        train_features, train_labels, test_size=0.05, random_state=42, stratify=train_labels
    )
    model, _ = train_multilayer_perceptron(
        holdout_training_features, holdout_training_labels,
        holdout_validation_features, holdout_validation_labels,
        hidden_layers=MLP_SUBMISSION_HIDDEN_LAYERS,
        dropout=MLP_SUBMISSION_DROPOUT_RATE,
        epochs=MLP_SUBMISSION_EPOCHS,
        learning_rate=MLP_SUBMISSION_LEARNING_RATE,
        activation=MLP_ACTIVATION
    )
    
    model.eval()
    
    with torch.no_grad():
        mlp_inference_device  = next(model.parameters()).device
        test_features_tensor  = torch.tensor(np.asarray(test_features, np.float32)).to(mlp_inference_device)
        predictions           = model(test_features_tensor).argmax(dim=1).cpu().numpy()

image_ids  = [os.path.splitext(os.path.basename(path))[0] for path in test_data['paths']]
submission = pd.DataFrame({'image_id': image_ids, 'class_id': predictions})

submission.to_csv('t1_submission.csv', index=False)
print(f'Saved {len(submission)} predictions to t1_submission.csv')

Training mlp on all 3750 labelled samples...
Saved 1250 predictions to t1_submission.csv
